# Load

In [8]:
from pathlib import Path
import pandas as pd

p = "data_anon/"

df_demographics = pd.read_csv(p + "admin_demographic_survey.csv")
df_exit = pd.read_csv(p + "admin_exit_survey.csv")
df_interactions = pd.read_csv(p + "admin_interactions.csv")
df_feedback = pd.read_csv(p + "admin_meeting_feedback.csv")
df_ranking = pd.read_csv(p + "llm_ranking_gpt-5.4_low.csv")
df_merged = pd.read_csv(p + "merged_by_ty.csv")
df_second_first = pd.read_csv(p + "second-first.csv")

df_control_diff = pd.read_csv(p + "survey_control_w_diff.csv")
df_control = pd.read_csv(p + "survey_control.csv")
df_survey_filtered = pd.read_csv(p + "survey_filtered.csv")
df_sophie_control_diff = pd.read_csv(p + "survey_sophie_control_w_diff.csv")
df_sophie_diff = pd.read_csv(p + "survey_sophie_w_diff.csv")
df_sophie = pd.read_csv(p + "survey_sophie.csv")

In [7]:
for file in Path("data_anon").glob("sophie_1.0_*.csv"):
    df = pd.read_csv(file)
    df[["content"]] = "[Redacted]"
    df.to_csv(file, index=False)

In [12]:
df_demographics.columns

Index(['user_id', 'email_m3_id', 'user_group', 'timestamp', 'age', 'gender',
       'sex_assigned_at_birth', 'race', 'ethnicity', 'childhood_social_class',
       'clinical_experience', 'education_completion_date', 'urmc_act_training',
       'other_communication_training', 'interacted_with_sophie_before'],
      dtype='object')

In [13]:
import json
from pathlib import Path

mapping_file = Path("user_group_mapping.json")

# 1. Create and save mapping
groups = sorted(df_demographics["user_group"].dropna().astype(str).unique())
user_group_mapping = {
    group: f"GROUP_{i:03d}"
    for i, group in enumerate(groups, start=1)
}

with mapping_file.open("w", encoding="utf-8") as f:
    json.dump(user_group_mapping, f, indent=2)

In [14]:
# 2. Replace user_group values in any DataFrame
def map_user_groups(df, column="user_group"):
    with mapping_file.open("r", encoding="utf-8") as f:
        mapping = json.load(f)

    result = df.copy()
    present = result[column].dropna().astype(str)
    unknown = sorted(set(present) - set(mapping))

    if unknown:
        raise ValueError(f"Unmapped user groups: {unknown}")

    mask = result[column].notna()
    result.loc[mask, column] = present.map(mapping)
    return result

In [53]:
import pandas as pd

def anonymize_df(df):
    result = df.copy()

    if "user_id" in result.columns:
        # Keep rows whose user_id can be interpreted as a number
        numeric_id = pd.to_numeric(result["user_id"], errors="coerce").notna()
        result = result.loc[numeric_id].copy()

    # Redact columns only when they exist
    columns_to_redact = [
        "email_m3_id",
        "recommend_email_text",
        "meeting_transcript",
        "recommend_email_text",
        "turns",
        "user_fname",
        "user_lname",
        "student_email",
        "meeting_transcript",
        "meeting_id",
        "transcript",
        "content"
    ]
    
    for column in columns_to_redact:
        if column in result.columns:
            result[column] = "[Redacted]"

    # Apply the previously defined user-group mapping
    if "user_group" in result.columns:
        result = map_user_groups(result)

    return result

In [54]:
from pathlib import Path
import pandas as pd

def anonymize_csv(filename):
    path = Path(filename)
    df = pd.read_csv(path)
    df = anonymize_df(df)
    df.to_csv(path, index=False)
    print(f"Updated: {path}")

In [55]:
anonymize_csv(p+'3E_by_llm2.csv')

Updated: data_anon\3E_by_llm2.csv
